# Gradient Descent by Hand

**MST374 — Tutorial notebook**

---

In the lecture slides, gradient descent was introduced as the engine that powers neural network training. Keras and Adam handle all of this automatically in the main computer sessions — but in this notebook we are going to do it **by hand**, with nothing but NumPy, so that you can see exactly what is happening at every step.

We will follow the two worked examples from the slides precisely.

---

## 1. The network we are working with

We use the tiny network sketched on slide 7 of the lecture. It has:

- **One scalar input** $x$
- **Two hidden nodes** $h_1$ and $h_2$, each with their own weight and bias connecting from $x$
- **One output node** $\hat{y}$, combining both hidden nodes
- **Sigmoid activation** $\sigma$ applied at every node

The weights are $\omega_1, \omega_2, \omega_3, \omega_4$ and the biases are $b_1, b_2$. Here is how the signal flows step by step:

$$a_1 = \omega_1 x - b_1, \qquad h_1 = \sigma(a_1)$$

$$a_2 = \omega_2 x - b_2, \qquad h_2 = \sigma(a_2)$$

$$a_{\text{out}} = \omega_3 h_1 + \omega_4 h_2, \qquad \hat{y} = \sigma(a_{\text{out}})$$

So the entire network is a function $f : \mathbb{R} \to \mathbb{R}$:

$$f(x) = \sigma\!\Bigl(\omega_3 \cdot \sigma(\omega_1 x - b_1) + \omega_4 \cdot \sigma(\omega_2 x - b_2)\Bigr)$$

This is exactly the formula from slide 7 of the lecture.

---

## 2. The loss function

Given training examples $\mathcal{D} = \{(x_i, y_i)\}$ where $y_i$ is the desired (true) output, we define the **mean squared error** (MSE) loss:

$$J = \frac{1}{|\mathcal{D}|} \sum_{(x,y) \in \mathcal{D}} \left(f(x) - y\right)^2$$

As the slide says: *this is really a function of the weights and biases*. Given fixed training data, we want to find the values of $\omega_1, \omega_2, \omega_3, \omega_4, b_1, b_2$ that make $J$ as small as possible. That is the whole point of training.

---

## 3. The gradient descent update rule

We cannot find the global minimum of $J$ analytically — it is a complicated, high-dimensional surface. Instead we use **gradient descent**: start somewhere on that surface, find the steepest downhill direction, and take a small step in that direction. Repeat until the loss stops decreasing.

For every parameter $\omega_i$, the update rule (from slide 6) is:

$$\boxed{\omega_{i,\,\text{new}} = \omega_{i,\,\text{old}} - \eta \, \frac{\partial J}{\partial \omega_i}}$$

where $\eta > 0$ is the **learning rate** — how big a step we take.

- If $\partial J / \partial \omega_i > 0$: $J$ increases as $\omega_i$ increases, so we **decrease** $\omega_i$.
- If $\partial J / \partial \omega_i < 0$: $J$ decreases as $\omega_i$ increases, so we **increase** $\omega_i$.

Either way, subtracting $\eta$ times the gradient moves us downhill. The same rule applies to every bias.

---

## 4. Computing the gradients — the chain rule

To apply the update rule we need to compute $\partial J / \partial \omega_i$ for each weight. Because $J$ depends on $\omega_i$ only *through* intermediate quantities (the node activations), we use the **chain rule**. The slides call this a *diagram of influence*.

We work through a **single training example** $(x, y)$ so that $J = (\hat{y} - y)^2$.

### 4.1 The derivative of sigmoid

The sigmoid function is $\sigma(z) = \dfrac{1}{1+e^{-z}}$. Its derivative is:

$$\sigma'(z) = \sigma(z)\,(1 - \sigma(z))$$

This appears everywhere in the backpropagation formulas below.

### 4.2 Gradient with respect to $\omega_3$ (output-layer weight on $h_1$)

The chain of influence from the slides (slide 9) is:

$$\omega_3 \;\longrightarrow\; a_{\text{out}} \;\longrightarrow\; \hat{y} \;\longrightarrow\; J$$

Applying the chain rule:

$$\frac{\partial J}{\partial \omega_3} = \frac{\partial J}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial a_{\text{out}}} \cdot \frac{\partial a_{\text{out}}}{\partial \omega_3}$$

Each factor:

$$\frac{\partial J}{\partial \hat{y}} = 2(\hat{y} - y), \qquad \frac{\partial \hat{y}}{\partial a_{\text{out}}} = \hat{y}(1-\hat{y}), \qquad \frac{\partial a_{\text{out}}}{\partial \omega_3} = h_1$$

So:

$$\frac{\partial J}{\partial \omega_3} = 2(\hat{y} - y) \cdot \hat{y}(1-\hat{y}) \cdot h_1$$

This matches the calculation on slide 10 (*"with $a = \hat{y}$ ... $w_{3,\text{new}} = w_3 - \eta \cdot ...$"*).

### 4.3 Gradient with respect to $\omega_1$ (weight in the hidden layer feeding $h_1$)

Now the chain is longer — $\omega_1$ is further from the output:

$$\omega_1 \;\longrightarrow\; a_1 \;\longrightarrow\; h_1 \;\longrightarrow\; a_{\text{out}} \;\longrightarrow\; \hat{y} \;\longrightarrow\; J$$

$$\frac{\partial J}{\partial \omega_1} = \frac{\partial J}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial a_{\text{out}}} \cdot \frac{\partial a_{\text{out}}}{\partial h_1} \cdot \frac{\partial h_1}{\partial a_1} \cdot \frac{\partial a_1}{\partial \omega_1}$$

The new factors:

$$\frac{\partial a_{\text{out}}}{\partial h_1} = \omega_3, \qquad \frac{\partial h_1}{\partial a_1} = h_1(1-h_1), \qquad \frac{\partial a_1}{\partial \omega_1} = x$$

All together:

$$\frac{\partial J}{\partial \omega_1} = 2(\hat{y} - y) \cdot \hat{y}(1-\hat{y}) \cdot \omega_3 \cdot h_1(1-h_1) \cdot x$$

Notice the pattern: the output-layer error signal $2(\hat{y} - y)\cdot\hat{y}(1-\hat{y})$ gets **propagated back** through the network, multiplied at each junction by the local sigmoid derivative and the connecting weight. This is why training is called **backpropagation**.

The formulas for $\omega_2$, $\omega_4$, $b_1$, $b_2$ follow the same logic — we derive them in the code below.

---

## 5. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

---

## 6. Define the building blocks

In [ ]:
def sigmoid(z):
    """sigma(z) = 1 / (1 + exp(-z))."""
    return 1.0 / (1.0 + np.exp(-z))

def mse_loss(y_hat, y):
    """
    Mean squared error: J = mean( (y_hat - y)^2 ).
    Works for scalars or numpy arrays.
    """
    return np.mean((y_hat - y) ** 2)

---

## 7. Forward pass

The forward pass takes an input $x$ and the current parameter values, and returns the network output $\hat{y}$. We save every intermediate value because backpropagation will need them.

In [ ]:
def forward(x, w1, w2, w3, w4, b1, b2):
    """
    Forward pass for a single input x (scalar or numpy array).

    Network:
        x --w1,b1--> h1 = sigma(w1*x - b1) --w3--\
                                                   +--> a_out = w3*h1 + w4*h2 --> y_hat = sigma(a_out)
        x --w2,b2--> h2 = sigma(w2*x - b2) --w4--/

    Returns a dict of all intermediate values.
    """
    a1    = w1 * x - b1          # pre-activation of h1
    h1    = sigmoid(a1)          # hidden node 1

    a2    = w2 * x - b2          # pre-activation of h2
    h2    = sigmoid(a2)          # hidden node 2

    a_out = w3 * h1 + w4 * h2   # pre-activation of output
    y_hat = sigmoid(a_out)       # network output

    return {'a1': a1, 'h1': h1,
            'a2': a2, 'h2': h2,
            'a_out': a_out, 'y_hat': y_hat}

---

## 8. Backward pass (backpropagation)

The backward pass computes all six partial derivatives using the chain rule formulas derived in Section 4. We work from the output node back towards the input — hence *back*propagation.

Notice how `delta_out`, the error signal at the output node, is computed once and then reused for every parameter update. This is the core efficiency of backpropagation.

In [ ]:
def backward(x, y, cache, w3, w4):
    """
    Backward pass: compute dJ/d(each parameter) using the chain rule.

    Parameters
    ----------
    x      : network input (scalar or array)
    y      : true label (scalar or array)
    cache  : dict returned by forward()
    w3, w4 : output-layer weights (needed to propagate gradient back)

    Returns
    -------
    dict of gradients: dw1, dw2, dw3, dw4, db1, db2
    
    """
    y_hat = cache['y_hat']
    h1    = cache['h1']
    h2    = cache['h2']

    # ---- Output layer ----
    # dJ/d(y_hat) = 2*(y_hat - y)
    # d(y_hat)/d(a_out) = y_hat*(1-y_hat)   [sigmoid derivative]
    # Combined output error signal ("delta" at the output node):
    delta_out = 2.0 * (y_hat - y) * y_hat * (1.0 - y_hat)

    # dJ/dw3 = delta_out * dJ/da_out * da_out/dw3 = delta_out * h1
    # dJ/dw4 = delta_out * h2
    dw3 = np.mean(delta_out * h1)
    dw4 = np.mean(delta_out * h2)

    # ---- Propagate back through hidden node h1 ----
    # dJ/dh1 = delta_out * w3   (how much h1 affects a_out)
    # dh1/da1 = h1*(1-h1)       (sigmoid derivative at h1)
    delta_h1 = delta_out * w3 * h1 * (1.0 - h1)

    # da1/dw1 = x,  da1/db1 = -1
    dw1 = np.mean(delta_h1 * x)
    db1 = np.mean(delta_h1 * (-1.0))

    # ---- Propagate back through hidden node h2 ----
    delta_h2 = delta_out * w4 * h2 * (1.0 - h2)

    dw2 = np.mean(delta_h2 * x)
    db2 = np.mean(delta_h2 * (-1.0))

    return {'dw1': dw1, 'dw2': dw2, 'dw3': dw3, 'dw4': dw4,
            'db1': db1, 'db2': db2}

---

## 9. One gradient descent step

This function combines forward pass, loss computation, backward pass, and parameter update into a single call.

In [ ]:
def gradient_step(x, y, params, eta):
    """
    One gradient descent step: forward -> loss -> backward -> update.

    Parameters
    ----------
    x      : input(s)
    y      : true label(s)
    params : dict with keys w1, w2, w3, w4, b1, b2
    eta    : learning rate

    Returns
    -------
    (updated_params, loss_before_update)
    """
    w1, w2, w3, w4 = params['w1'], params['w2'], params['w3'], params['w4']
    b1, b2         = params['b1'], params['b2']

    cache = forward(x, w1, w2, w3, w4, b1, b2)
    loss  = mse_loss(cache['y_hat'], y)
    grads = backward(x, y, cache, w3, w4)

    # Apply the update rule: omega_new = omega_old - eta * dJ/domega
    new_params = {
        'w1': w1 - eta * grads['dw1'],
        'w2': w2 - eta * grads['dw2'],
        'w3': w3 - eta * grads['dw3'],
        'w4': w4 - eta * grads['dw4'],
        'b1': b1 - eta * grads['db1'],
        'b2': b2 - eta * grads['db2'],
    }
    return new_params, loss

---

## 10. Example 1 — single training case, one step by hand

This is the *"really easy example"* from slide 9 of the lecture.

We have a single training case: input $x = 0.5$, desired output $y = 1$. We pick initial weights by hand and carry out **exactly one step** of gradient descent, printing everything so you can verify the chain rule calculation with a calculator if you like.

After the step, the prediction should move closer to 1 and the loss should decrease.

In [ ]:
x_ex1 = 0.5
y_ex1 = 1.0

# Initial parameter values — small, as is standard at initialisation
params_ex1 = {'w1': 0.3, 'w2': -0.2, 'w3': 0.5, 'w4': 0.4,
              'b1': 0.1, 'b2':  0.05}
eta = 1.0

# --- Forward pass to see the initial state ---
c0 = forward(x_ex1, **params_ex1)

print('=== Before any training ===')
print(f"  a1     = w1*x - b1 = {params_ex1['w1']}*{x_ex1} - {params_ex1['b1']} = {c0['a1']:.4f}")
print(f"  h1     = sigma(a1) = {c0['h1']:.6f}")
print(f"  a2     = w2*x - b2 = {params_ex1['w2']}*{x_ex1} - {params_ex1['b2']} = {c0['a2']:.4f}")
print(f"  h2     = sigma(a2) = {c0['h2']:.6f}")
print(f"  a_out  = w3*h1 + w4*h2 = {c0['a_out']:.6f}")
print(f"  y_hat  = sigma(a_out)  = {c0['y_hat']:.6f}")
print(f"  loss J = (y_hat - y)^2 = {mse_loss(c0['y_hat'], y_ex1):.6f}")

# --- One gradient descent step ---
params_ex1_new, loss_before = gradient_step(x_ex1, y_ex1, params_ex1, eta)
c1 = forward(x_ex1, **params_ex1_new)

print()
print('=== After one gradient descent step (eta = 1.0) ===')
print(f"  y_hat  = {c1['y_hat']:.6f}  (was {c0['y_hat']:.6f})  [target: {y_ex1}]")
print(f"  loss J = {mse_loss(c1['y_hat'], y_ex1):.6f}  (was {loss_before:.6f})")
print()
print('Updated parameters:')
for k in ['w1','w2','w3','w4','b1','b2']:
    print(f"  {k}: {params_ex1[k]:+.4f}  -->  {params_ex1_new[k]:+.6f}")

**What to notice:**
- The initial prediction $\hat{y}$ is near 0.5 — the network starts knowing nothing.
- After one step, $\hat{y}$ is closer to 1.0 and the loss is lower.
- Every single weight and bias has moved — all six parameters were updated simultaneously.

---

## 11. Example 2 — many steps, watching the loss converge

One step is not enough. We now run the same network and training case for 500 steps and plot the loss history. The loss should fall steeply at first and then plateau — the network has learned to predict $y = 1$ given $x = 0.5$.

In [ ]:
# Reset to the original parameters
params = {'w1': 0.3, 'w2': -0.2, 'w3': 0.5, 'w4': 0.4,
          'b1': 0.1, 'b2':  0.05}
eta = 1.0
n_steps = 500
history = []

for _ in range(n_steps):
    params, loss = gradient_step(x_ex1, y_ex1, params, eta)
    history.append(loss)

c_final = forward(x_ex1, **params)
print(f"After {n_steps} steps: y_hat = {c_final['y_hat']:.6f}  (target: {y_ex1})")

plt.figure(figsize=(7, 4))
plt.plot(history)
plt.xlabel('Gradient descent step')
plt.ylabel('Loss $J$')
plt.title('Loss during training (single training case, $x=0.5$, $y=1$)')
plt.tight_layout()
plt.show()

---

## 12. A more interesting case — fitting a small dataset

A single example is trivial. Let's give the network five training points and watch it learn a pattern. We use the same two-hidden-node network, the same MSE loss, and vanilla gradient descent — nothing changes in the code at all, because it already handles batches via `np.mean` in the gradient computations.

In [ ]:
X_train = np.array([0.1, 0.3, 0.5, 0.7, 0.9])
Y_train = np.array([0.0, 0.0, 1.0, 1.0, 1.0])   # simple step-like pattern

# Fixed random seed for reproducibility
rng    = np.random.default_rng(374)
params = {k: rng.uniform(-0.5, 0.5) for k in ['w1','w2','w3','w4','b1','b2']}

print('Initial parameters:')
for k, v in params.items():
    print(f"  {k} = {v:+.4f}")

eta     = 2.0
n_steps = 2000
history = []

for _ in range(n_steps):
    params, loss = gradient_step(X_train, Y_train, params, eta)
    history.append(loss)

print(f"\nFinal loss after {n_steps} steps: {history[-1]:.6f}")

c_final = forward(X_train, **params)
print("\nPredictions vs targets:")
for xi, yi, yhat in zip(X_train, Y_train, c_final['y_hat']):
    print(f"  x={xi:.1f}  target={yi:.0f}  predicted={yhat:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: loss curve
axes[0].plot(history)
axes[0].set_xlabel('Gradient descent step')
axes[0].set_ylabel('MSE Loss $J$')
axes[0].set_title('Loss during training (5-point dataset)')

# Right: learned function vs training data
x_plot  = np.linspace(0.0, 1.0, 300)
c_plot  = forward(x_plot, **params)
axes[1].plot(x_plot, c_plot['y_hat'], label=r'Network output $\hat{y}$')
axes[1].scatter(X_train, Y_train, color='red', zorder=5, s=60, label='Training data')
axes[1].set_xlabel('$x$')
axes[1].set_ylabel('$\\hat{y}$')
axes[1].set_title('Learned function after training')
axes[1].legend()

plt.tight_layout()
plt.show()

The right-hand panel shows the **learned function**: the smooth S-shaped curve the network has fitted through the five red training points.

This is exactly what Keras does internally when you call `model.fit()`. The only practical differences are:
- Keras uses **Adam** instead of vanilla gradient descent (Adam adapts the learning rate per parameter).
- Keras processes the training data in **mini-batches** rather than the full dataset at once.
- Keras handles networks with many layers and many nodes automatically via autodifferentiation — but the chain rule mathematics is the same as what we coded above.

---

## 13. The effect of the learning rate $\eta$

The learning rate $\eta$ in the update rule $\omega_{\text{new}} = \omega_{\text{old}} - \eta \, \partial J / \partial \omega$ controls how large each step is:

- **Too small**: descent is correct but painfully slow.
- **Too large**: steps overshoot the minimum; loss oscillates or diverges.
- **Just right**: steady convergence in a reasonable number of steps.

The slide shows the loss surface and the yellow trajectory descending — the learning rate determines how far along that trajectory you move at each step.

In [ ]:
learning_rates = {
    r'Too small ($\eta = 0.05$)': 0.05,
    r'Good ($\eta = 2.0$)':       2.0,
    r'Too large ($\eta = 20.0$)': 20.0,
}

n_steps = 300
plt.figure(figsize=(9, 4))

for label, eta in learning_rates.items():
    rng    = np.random.default_rng(374)   # identical starting point each time
    params = {k: rng.uniform(-0.5, 0.5) for k in ['w1','w2','w3','w4','b1','b2']}
    history = []
    for _ in range(n_steps):
        params, loss = gradient_step(X_train, Y_train, params, eta)
        history.append(loss)
    plt.plot(history, label=label)

plt.xlabel('Step')
plt.ylabel('Loss $J$')
plt.title('Effect of learning rate on gradient descent')
plt.ylim(0, 0.35)
plt.legend()
plt.tight_layout()
plt.show()

---

## 14. Sanity check — numerical gradient verification

A standard way to check that a backpropagation implementation is correct is to compare the analytically computed gradients against **numerical gradients**, computed by finite differences:

$$\frac{\partial J}{\partial \omega_i} \approx \frac{J(\omega_i + \varepsilon) - J(\omega_i - \varepsilon)}{2\varepsilon}$$

for a very small $\varepsilon$. If the chain rule formulas are correct, the two should agree to many decimal places.

In [ ]:
def numerical_gradient(x, y, params, key, eps=1e-5):
    """Central-difference numerical gradient for one parameter."""
    p_plus  = dict(params); p_plus[key]  += eps
    p_minus = dict(params); p_minus[key] -= eps

    def loss_for(p):
        c = forward(x, **p)
        return mse_loss(c['y_hat'], y)

    return (loss_for(p_plus) - loss_for(p_minus)) / (2 * eps)

# Fresh arbitrary parameters for the check
rng         = np.random.default_rng(42)
check_p     = {k: rng.uniform(-1, 1) for k in ['w1','w2','w3','w4','b1','b2']}

# Analytical
c_chk       = forward(X_train, **check_p)
analytic    = backward(X_train, Y_train, c_chk, check_p['w3'], check_p['w4'])

print(f"{'Param':>6}   {'Analytic':>15}   {'Numerical':>15}   {'|Difference|':>15}")
print('-' * 62)
for key in ['w1','w2','w3','w4','b1','b2']:
    a = analytic['d' + key]
    n = numerical_gradient(X_train, Y_train, check_p, key)
    print(f"{key:>6}   {a:>15.8f}   {n:>15.8f}   {abs(a-n):>15.2e}")

The differences should all be on the order of $10^{-9}$ or smaller — numerical noise only. This confirms that the chain rule formulas in Section 4 and the `backward()` function are correct.

---

## 15. Summary: from this notebook to Keras

| Step | What we did here | Keras equivalent |
|------|-----------------|------------------|
| Define network topology | `forward()` | `model = Sequential(...)` with `Dense` layers |
| Choose loss | `mse_loss()` | `loss='mse'` in `model.compile()` |
| Compute gradients | `backward()` via chain rule | Automatic differentiation (TensorFlow) |
| Update weights | `w -= eta * grad` | `optimizer=SGD(learning_rate=eta)` |
| Repeat over steps | `for _ in range(n_steps)` | `model.fit(..., epochs=N)` |
| Adaptive step size | fixed $\eta$ | `optimizer=Adam()` |

The Heathrow network in notebook `10SML_03` has 11 inputs, two hidden layers of 5 nodes each, and one output — a total of $11 \times 5 + 5 + 5 \times 5 + 5 + 5 \times 1 + 1 = 91$ parameters. Keras computes all 91 gradients automatically using the same chain rule extended to many layers, then updates them simultaneously every mini-batch.

---

## Exercise

The network above has one input feature. Modify it to accept **two** input features $x_1$ and $x_2$, so that the hidden node pre-activations become:

$$a_1 = \omega_1 x_1 + \omega_5 x_2 - b_1, \qquad a_2 = \omega_2 x_1 + \omega_6 x_2 - b_2$$

You will need to:

1. Add weights `w5` and `w6` to `forward()`.
2. Add the corresponding gradients `dw5` and `dw6` to `backward()`, using $\partial a_1 / \partial \omega_5 = x_2$.
3. Train on the following dataset and verify that the loss decreases:

```python
X2 = np.array([[0.1, 0.8],
               [0.3, 0.6],
               [0.5, 0.5],
               [0.7, 0.3],
               [0.9, 0.1]])
Y2 = np.array([0.0, 0.0, 1.0, 1.0, 1.0])
```

This extension mirrors the structure of the Heathrow MLP, where each of the 11 input nodes connects to each hidden node with its own weight.